# Chaotic Saddle Computations (Sprinkler Method)

This notebook starts by testing the new `return_mod_period` flag in `FastSitnikovSimulation.phi_fast` for `e=0.5`.

In [2]:
from pathlib import Path
import sys
import numpy as np

# Make project root importable from this experiment folder
ROOT = Path.cwd().parents[2]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.integrator.integrate import FastSitnikovSimulation

sim = FastSitnikovSimulation(e=0.5, solver_method='RK45')
sim

In [3]:
# Validate that raw time and modulo time are consistent
t0 = 20.0 * np.pi + 0.37
candidate_velocities = np.linspace(0.15, 1.2, 14)

hit = None
for v0 in candidate_velocities:
    v_mod, t_mod = sim.phi_fast(v0, t0, t_max=60.0, return_mod_period=True)
    v_raw, t_raw = sim.phi_fast(v0, t0, t_max=60.0, return_mod_period=False)
    if v_mod is not None and v_raw is not None:
        hit = (v0, v_mod, t_mod, v_raw, t_raw)
        break

if hit is None:
    raise RuntimeError('No returning orbit found in candidate velocity set; increase t_max or widen velocity scan.')

v0, v_mod, t_mod, v_raw, t_raw = hit
print(f'Using v0 = {v0:.6f}')
print(f'v_mod = {v_mod:.12f}, v_raw = {v_raw:.12f}')
print(f't_mod = {t_mod:.12f}, t_raw = {t_raw:.12f}')
print(f't_raw % (2pi) = {np.mod(t_raw, 2.0*np.pi):.12f}')

assert np.isclose(v_mod, v_raw, rtol=1e-10, atol=1e-12), 'Velocity mismatch between mod/raw modes'
assert np.isclose(t_mod, np.mod(t_raw, 2.0*np.pi), rtol=1e-10, atol=1e-12), 'Time modulo relation failed'
assert t_raw >= t0, 'Raw crossing time should be absolute and not wrapped'
print('Flag test passed.')

Using v0 = 0.150000
v_mod = 0.098174254053, v_raw = 0.098174254053
t_mod = 1.197396366497, t_raw = 64.029249438293
t_raw % (2pi) = 1.197396366497
Flag test passed.


## Step 1: Fixed Parameters and Deterministic Setup

We now implement the first sprinkler-method pilot with:
- `e = 0.5`
- `t0 = 0`
- `t_max = 20*pi`
- strict `>100` crossings checked via `max_crossings=101`
- deterministic RNG seed

In [5]:
from src.utils.boundary import B1_v_func

E = 0.5
T0 = 0.0
N0 = 1000
N_STEPS = 40
SEED = 20260406

rng = np.random.default_rng(SEED)

sim = FastSitnikovSimulation(e=E, solver_method='RK45')

print('Configured deterministic pilot:')
print(f'  E = {E}')
print(f'  T0 = {T0}')
print(f'  N0 = {N0}, N_STEPS = {N_STEPS}, seed = {SEED}')

Configured deterministic pilot:
  E = 0.5
  T0 = 0.0
  N0 = 1000, N_STEPS = 40, seed = 20260406


In [6]:
# Compute v_max from the boundary at t=0, then find v_min from the >100-crossings rule
# Use a moderate boundary sampling to keep this interactive while still using B1_v_func.
v_escape_t0 = float(B1_v_func(E, dv=2e-3, N_t=40)(0.0))
v_max = v_escape_t0 - 0.02

print(f'v_max from boundary at t=0: {v_max:.12f}')

v_max from boundary at t=0: 2.745213574142


In [ ]:
N_min_test = 31
N_coarse_grid = 10
N_fine_grid = 10

k_offsets = 1e-2 * np.array([5, 4, 3, 2, 1, 0], dtype=float)

def qualifies_for_vmin(v_candidate):
    for off in k_offsets:
        v_test = v_candidate - off
        if v_test <= 0.0:
            return False
        n_cross = sim.crossings_fast(v_test, T0, max_crossings=N_min_test)
        if n_cross < N_min_test:
            return False
    return True

# Deterministic 2-stage scan for the highest qualifying v*
# User-informed bracket for faster search
v_scan_lo = 1.2
v_scan_hi = 2.2
if v_scan_hi <= v_scan_lo:
    raise RuntimeError(
        f'Invalid scan interval after boundary cap: [{v_scan_lo:.6f}, {v_scan_hi:.6f}]'
    )

coarse_grid = np.linspace(v_scan_lo, v_scan_hi, N_coarse_grid)
v_star_coarse = None
for vv in coarse_grid:
    if qualifies_for_vmin(vv):
        v_star_coarse = vv
    else:
        break

if v_star_coarse is None:
    raise RuntimeError('No qualifying v* found in coarse scan; expand search interval or adjust t_max.')

coarse_step = coarse_grid[1] - coarse_grid[0]
fine_lo = v_star_coarse - coarse_step
fine_hi = v_star_coarse + coarse_step
fine_grid = np.linspace(fine_lo, fine_hi, N_fine_grid)
fine_ok = np.array([qualifies_for_vmin(vv) for vv in fine_grid], dtype=bool)

if not np.any(fine_ok):
    raise RuntimeError('Fine scan failed near coarse optimum; this is unexpected.')

v_star = float(fine_grid[fine_ok][-1])
v_min = v_star + 0.05

print(f'v_escape(t=0) = {v_escape_t0:.12f}')
print(f'v_max        = {v_max:.12f}')
print(f'v_star       = {v_star:.12f}')
print(f'v_min        = {v_min:.12f}')
print(f'interval ok? {v_min < v_max}')

if not (v_min < v_max):
    raise RuntimeError(
        f'Computed interval is empty: v_min={v_min:.6f} >= v_max={v_max:.6f}. '
        'Need to revisit margin definitions or search strategy.'
    )

v_escape(t=0) = 2.765213574142
v_max        = 2.745213574142
v_star       = 2.200000000000
v_min        = 2.250000000000
interval ok? True


In [7]:
# Pilot sprinkler run on R = {(t=0, v): v in (v_min, v_max)}
# Survivors at n0 approximate points near the stable manifold in initial space,
# midpoint survivors approximate the saddle, and final survivors approximate unstable directions.

v0_samples = rng.uniform(v_min, v_max, size=N0)
mid_step = N_STEPS // 2

survivor_initial_v = []
survivor_mid_t = []
survivor_mid_v = []
survivor_final_t = []
survivor_final_v = []
escape_step = np.full(N0, N_STEPS + 1, dtype=int)

for i, v_init in enumerate(v0_samples):
    t_curr = T0
    v_curr = float(v_init)
    mid_state = None
    survived = True

    for step in range(1, N_STEPS + 1):
        v_next, t_next = sim.phi_fast(v_curr, t_curr, t_max=T_MAX, return_mod_period=True)
        if v_next is None:
            escape_step[i] = step
            survived = False
            break

        v_curr = float(v_next)
        t_curr = float(t_next)

        if step == mid_step:
            mid_state = (t_curr, v_curr)

    if survived:
        survivor_initial_v.append(float(v_init))
        if mid_state is None:
            raise RuntimeError('Missing midpoint state for a surviving trajectory.')
        survivor_mid_t.append(mid_state[0])
        survivor_mid_v.append(mid_state[1])
        survivor_final_t.append(t_curr)
        survivor_final_v.append(v_curr)

survivor_initial_v = np.array(survivor_initial_v)
survivor_mid_t = np.array(survivor_mid_t)
survivor_mid_v = np.array(survivor_mid_v)
survivor_final_t = np.array(survivor_final_t)
survivor_final_v = np.array(survivor_final_v)

print(f'Survivors after n0={N_STEPS}: {survivor_initial_v.size}/{N0}')
print(f'Estimated survival fraction: {survivor_initial_v.size / N0:.4f}')
print(f'Min/Max finite escape step: {escape_step.min()} / {escape_step[escape_step <= N_STEPS].max() if np.any(escape_step <= N_STEPS) else "none"}')

NameError: name 'v_min' is not defined